# Day 2: Building a Real-World Telecom RAG System

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

# Multilingual embeddings (Crucial for matching Arabic queries to English text)
print("Loading local embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Load the Knowledge Base
print("Loading Knowledge Base...")
loader = TextLoader('data/Telecom_Internal_KB.txt', encoding='utf-8')
documents = loader.load()
print(f"✅ Successfully loaded {len(documents)} document(s).")

Loading local embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3693.90it/s]


Loading Knowledge Base...
✅ Successfully loaded 1 document(s).


## Structure-Aware Markdown Chunking
Improve the original chunking strategy by using the knowledge base's existing Markdown hierarchy (`#`, `##`, `###`) to split content along meaningful section boundaries instead of relying only on character length. This preserves the context and metadata of related sections, such as keeping a router model's specifications logically grouped, while recursive splitting is used afterward only for sections that are still too large.

In [ ]:
# Split the text into chunks
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

# Step 1: Split by headers (# ## ###) first
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

# Apply it to the document text (not directly to the documents, it needs the text as a string)
md_chunks = markdown_splitter.split_text(documents[0].page_content)

# Step 2: Any section that is still too large (like a long SLA section) is split into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)
chunks = text_splitter.split_documents(md_chunks)

print(f"✅ Split into {len(chunks)} chunks.")
print(f"🔍 Sample chunk metadata: {chunks[1].metadata}")   # You will find the header name itself here!
print(f"🔍 Sample chunk content: \n{chunks[2].page_content}")

✅ Split into 490 chunks.
🔍 Sample chunk metadata: {'Header 1': 'Telecom Egypt Internal Technical Support Knowledge Base (Confidential)', 'Header 2': '2. Hardware Specifications & Router Guides', 'Header 3': 'Router Model: VDF-ZTE-2023X1'}
🔍 Sample chunk content: 
- **Troubleshooting Step 1:** Restart router and wait 2 minutes.
- **Troubleshooting Step 2:** Factory reset by holding the reset pin for 10 seconds. Reconfigure with VLAN ID 20.


In [ ]:
from langchain_community.vectorstores import FAISS
from tqdm import tqdm

print(f"Starting ingestion of {len(chunks)} chunks into FAISS...")

# We ingest in batches to lower resource costs
batch_size = 50
vectorstore = None

for i in tqdm(range(0, len(chunks), batch_size), desc="Embedding & Indexing Chunks"):
    batch = chunks[i:i + batch_size]
    
    if vectorstore is None:
        # First batch initializes the FAISS index
        vectorstore = FAISS.from_documents(batch, embeddings)
    else:
        # Subsequent batches are added to the existing index
        vectorstore.add_documents(batch)
        

# Save the FAISS index locally so we don't have to pay/wait to re-embed later
vectorstore.save_local("faiss_telecom_index")
print("\n✅ Ingestion Complete. FAISS index saved locally.")

# Test if the multilingual retrieval actually works!
print("\nTesting semantic search (Arabic Query -> English Document)...")
test_query = "العميل بيشتكي إن لمبة الراوتر بتنور وتطفي بقالها ٣ أيام"
print(f"\n{test_query}")

results = vectorstore.similarity_search_with_score(test_query, k=4)
print("\n✅ Top matches retrieved by FAISS:")
print("--------------------------------------------------")

for rank, (doc, score) in enumerate(results, start=1):
    source_section = doc.metadata.get("Header 3", doc.metadata.get("Header 2", "N/A"))
    print(f"#{rank} | Match Score: {score:.4f} | Source Section: {source_section}")
    print(doc.page_content)
    print("--------------------------------------------------")

Starting ingestion of 490 chunks into FAISS...


Embedding & Indexing Chunks: 100%|██████████| 10/10 [00:15<00:00,  1.54s/it]


✅ Ingestion Complete. FAISS index saved locally.

Testing semantic search (Arabic Query -> English Document)...

العميل بيشتكي إن لمبة الراوتر بتنور وتطفي بقالها ٣ أيام

✅ Top matches retrieved by FAISS:
--------------------------------------------------
#1 | Match Score: 12.1922 | Source Section: 4. Cross-Department Escalation Matrix
- **Billing Issues:** Transfer to 111.
- **Fiber Optic Cuts:** Escalate immediately to Tier 3 Fiber Ops. SLA is 12 hours.
- **Mass Outage (Area Level):** Do not dispatch individual technicians. Read the 'Global Outage Script' to the customer.
--------------------------------------------------
#2 | Match Score: 12.6330 | Source Section: 1. General Service Level Agreement (SLA) & Dispatch Policies
If a customer reports an internet outage (DSL blinking or no sync):
- The L1 agent must first ensure the customer has restarted the router and checked internal wiring.
- If the issue persists for more than 24 hours, the L1 agent must escalate to the Central Excha

## Metadata-Aware Context Formatting & Optimized Retrieval

Make the RAG pipeline actually use the metadata created during Markdown-based chunking by adding the section name to every retrieved chunk before sending the context to Gemini. This gives the LLM a clear indication of where each piece of information comes from and helps prevent confusion between similar router models. Additionally, replace the original `top-k` retrieval with MMR and reduce the number of final results from 20 to 6, since the new chunks are more complete and information-dense. MMR selects diverse and relevant results from a larger candidate pool (`fetch_k=20`), reducing redundant context, unnecessary noise, and token usage while keeping the retrieved context focused on the customer's issue.

## Prompt Engineering & Response Quality Constraints
Strengthen the system prompt with explicit style and safety constraints to make Gemini behave more like a real Egyptian customer service agent. The prompt prevents company-name leakage, unsupported information, fabricated answers, and overly formal language, while keeping responses concise (around 4–5 sentences) and avoiding repetitive openings for a more natural and consistent customer experience.

In [ ]:
import os
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

print("Building the Prompt and Gemini RAG Chain...")

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("Error: GOOGLE_API_KEY is not found in the .env file — make sure it is added correctly.")
os.environ["GOOGLE_API_KEY"] = api_key

gemini_llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)

template = """
أنت موظف خدمة عملاء في مزود خدمة إنترنت (ISP).
مهمتك هي الرد على شكوى العميل بالعامية المصرية بطريقة مهذبة واحترافية.
ممنوع تمامًا:
- ذكر أي اسم شركة اتصالات حقيقي
- ذكر أنك ذكاء اصطناعي أو بوت
- إعطاء رقم/إيميل/رابط غير موجود حرفيًا في السياق
- اختراع معلومة غير موجودة في السياق؛ حوّلها لفريق مختص لو ناقصة
- استخدام ألفاظ رسمية زي "سيادتكم" أو "حضرتكم الموقر"
- تكرار نفس جملة الافتتاح في كل رد
الرد يكون في حدود 4-5 جمل بس، من غير حشو.
السياق الداخلي:
{context}
شكوى العميل:
{question}
الرد:
"""

prompt = PromptTemplate.from_template(template)

# Improvement 1: Use MMR instead of regular top-k, with a smaller k since the chunks are now more information-dense
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 6, "fetch_k": 20}
)

# Improvement 2: Add the section name for each chunk to the context
def format_docs(docs):
    formatted = []
    for doc in docs:
        section = doc.metadata.get("Header 3") or doc.metadata.get("Header 2", "General")
        formatted.append(f"[من قسم: {section}]\n{doc.page_content}")
    return "\n\n".join(formatted)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | gemini_llm | StrOutputParser()
)
print("✅ Gemini RAG Chain is ready!")

Building the Prompt and Gemini RAG Chain...
✅ Gemini RAG Chain is ready!


In [14]:
print("Processing the ticket through Gemini...\n")

customer_ticket = """
أنا دافع الفاتورة من يومين أونلاين والفلوس اتخصمت من الفيزا، 
لكن النت لسه مرجعش لحد دلوقتي ومكتوبلي إن الخدمة موقوفة!
"""

print("Agent AI Response (Gemini):")
print("--------------------------------------------------")

# This sends the ticket to the retriever, formats the prompt, and gets the answer from Gemini
response = rag_chain.invoke(customer_ticket)

print(response)
print("--------------------------------------------------")

Processing the ticket through Gemini...

Agent AI Response (Gemini):
--------------------------------------------------


c:\Users\Eslam\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


مساء الخير يا فندم، بعتذر لحضرتك جدًا عن الإزعاج ده ومقدر موقفك تمامًا. 

بالنسبة لمشاكل الفواتير وعمليات الدفع الإلكتروني، بيتم تحويلها فورًا لقسم الحسابات على رقم 111. زمايلنا في القسم هيراجعوا عملية السداد وتأكيد الخصم عشان يتم إعادة تفعيل الخدمة لحضرتك على طول. تقدر تتواصل معاهم حالاً على 111 لمتابعة المشكلة وحلها في أسرع وقت.
--------------------------------------------------


In [15]:
print("Processing the ticket through Gemini...\n")

customer_ticket = """
النت شغال بس بطيء جداً وبيظهرلي رسالة على الشاشة فيها كود الخطأ E-204.
أعمل إيه عشان أحل المشكلة دي؟
"""

print("Agent AI Response (Gemini):")
print("--------------------------------------------------")

# This sends the ticket to the retriever, formats the prompt, and gets the answer from Gemini
response = rag_chain.invoke(customer_ticket)

print(response)
print("--------------------------------------------------")

Processing the ticket through Gemini...

Agent AI Response (Gemini):
--------------------------------------------------


c:\Users\Eslam\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


أهلاً بك، متأسف جداً على المشكلة والبطء اللي بتواجهه في النت. كود الخطأ E-204 بيظهر لما يكون فيه تشويش مرتفع على الخط. عشان نحل المشكلة دي، ياريت تغير عنوان الـ DNS في إعدادات الجهاز أو الراوتر لـ 8.8.8.8. جرب الخطوة دي وقولي لو السرعة تحسنت معاك. ولو محتاج أي مساعدة في تطبيق الخطوات أنا معاك خطوة بخطوة.
--------------------------------------------------
